In [ ]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, SVD, accuracy
from surprise.model_selection import train_test_split


In [ ]:
data = Dataset.load_builtin('ml-100k', prompt=False)

home_dir=os.path.expanduser('~')
items_path=os.path.join(home_dir,'.surprise_data', 'ml-100k', 'ml-100k', 'u.item')

In [ ]:
def load_and_train_model():
    data = Dataset.load_builtin('ml-100k')

    movie_titles = {}
    home_dir = os.path.expanduser('~')
    items_path = os.path.join(home_dir, '.surprise_data', 'ml-100k', 'ml-100k', 'u.item')

    if os.path.exists(items_path):
        with open(items_path, 'r', encoding='ISO-8859-1') as f:
            for line in f:
                fields = line.split('|')
                movie_titles[fields[0]] = fields[1]

    trainset, testset = train_test_split(data, test_size=0.25, random_state=42)
    model = SVD(n_factors=50, n_epochs=25, lr_all=0.005, reg_all=0.02, random_state=42)
    model.fit(trainset)

    predictions = model.test(testset)
    rmse_score = accuracy.rmse(predictions, verbose=False)


    return model, trainset, movie_titles, rmse_score


In [ ]:
def get_top_n_recommendations(user_id, model, trainset, movie_titles, n=5):
    user_id = str(user_id)
    all_movie_ids = list(movie_titles.keys())

    try:
        inner_user_id = trainset.to_inner_uid(user_id)
        user_rated_items = set([item_id for (item_id, rating) in trainset.ur[inner_user_id]])
    except ValueError:
        user_rated_items = set()

    predictions = []
    for movie_id in all_movie_ids:
        try:
            inner_item_id = trainset.to_inner_iid(movie_id)
            if inner_item_id in user_rated_items:
                continue
        except ValueError:
            pass

        pred = model.predict(user_id, movie_id)
        predictions.append({
            'Tên Phim': movie_titles.get(movie_id, "Unknown"),
            'Điểm Dự Đoán': round(pred.est, 2)
        })

    predictions.sort(key=lambda x: x['Điểm Dự Đoán'], reverse=True)
    return predictions[:n]